In [7]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
import requests

In [8]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [9]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetches current weather data for a given city using the Open-Meteo API.

    This function performs the following steps:
    1. Calls the Open-Meteo Geocoding API to convert the city name into latitude and longitude.
    2. Uses the obtained coordinates to fetch current weather data from the Open-Meteo Forecast API.

    Args:
        city (str): Name of the city for which weather data is required.

    Returns:
        str: JSON response containing current weather details such as temperature,
             wind speed, wind direction, and timestamp.
    """
    # Step 1: Get coordinates
    geocode_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
    geo_response = requests.get(geocode_url).json()
    lat = geo_response['results'][0]['latitude']
    lon = geo_response['results'][0]['longitude']

    # Step 2: Get current weather
    weather_url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "current_weather": True
    }
    weather_response = requests.get(weather_url, params=params).json()
    return weather_response

In [10]:
# Test
print(get_weather_data.invoke({
    "city": "Bhubaneswar"
}))

{'latitude': 20.25, 'longitude': 85.875, 'generationtime_ms': 0.06556510925292969, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 59.0, 'current_weather_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature': '°C', 'windspeed': 'km/h', 'winddirection': '°', 'is_day': '', 'weathercode': 'wmo code'}, 'current_weather': {'time': '2026-03-18T17:30', 'interval': 900, 'temperature': 24.7, 'windspeed': 5.4, 'winddirection': 188, 'is_day': 0, 'weathercode': 3}}


In [11]:
## LLM creation
llm = ChatOllama(model="qwen3-vl:4b")

In [12]:
from langchain_classic.agents.agent import AgentExecutor
from langchain_classic.agents import create_react_agent

In [19]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt("hwchase17/react")

prompt.template += """

IMPORTANT:
- Do NOT repeat the same action multiple times.
- If you already found the capital, move to next step.
- Always provide Final Answer when done.
"""

In [20]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

d:\Machine Learning Revision\AgenticAI\YouTube_CampusX\LangChain-Learning\venv\Lib\site-packages\langchain_core\runnables\utils.py:756: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Python 3.16; use inspect.iscoroutinefunction() instead
  return asyncio.iscoroutinefunction(func) or (
d:\Machine Learning Revision\AgenticAI\YouTube_CampusX\LangChain-Learning\venv\Lib\site-packages\langchain_core\runnables\utils.py:758: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Python 3.16; use inspect.iscoroutinefunction() instead
  and asyncio.iscoroutinefunction(func.__call__)


In [21]:
# Step 4: Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3   # reduce looping
)

In [22]:
# Step 5: Invoke
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})
print(response)



> Entering new AgentExecutor chain...
Invalid Format: Missing 'Action:' after 'Thought:'Action: duckduckgo_search  
Action Input: capital of Madhya PradeshOften referred to as the HeartofIndia,MadhyaPradeshis a state in central India ... Bhopal is thecapitalofthe Indian stateofMadhyaPradesh. Less-fertile red-to-yellow soils are spread over muchofeasternMadhyaPradesh. ...MadhyaPradeshhas a numberofnational parks and many wildlife ... MadhyaPradeshstate is established on 1-Nov-1956 and thecapitalofMadhyaPradeshis Bhopal. ... Total numberofhouseholds inMadhyaPradeshState ... MadhyaPradeshstate is made upof48 districts, which are grouped into nine divisions: Bhopal , Chambal, Gwalior , Hoshangabad, Indore , Jabalpur ... MadhyaPradeshwas created in 1950 from the former British Central Provinces and Berar and the princely statesofMakrai and Chhattisgarh, with Nagpur ...Thought: I now know the final answer  
Final Answer: The capital of Madhya Pradesh is Bhopal. The current weather in Bhopa

## Modern tool based approach without using ReAct

In [26]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a helpful assistant. "
     "You can use tools to answer questions. "
     "Use the appropriate tool when needed. "
     "If multiple steps are required, complete them step by step."
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [29]:
from langchain_classic.agents.tool_calling_agent.base import create_tool_calling_agent

tools = [search_tool, get_weather_data]

agent = create_tool_calling_agent(llm, tools, prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3   # reduce looping
)

response = executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})
print(response)

d:\Machine Learning Revision\AgenticAI\YouTube_CampusX\LangChain-Learning\venv\Lib\site-packages\langchain_core\runnables\utils.py:756: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Python 3.16; use inspect.iscoroutinefunction() instead
  return asyncio.iscoroutinefunction(func) or (
d:\Machine Learning Revision\AgenticAI\YouTube_CampusX\LangChain-Learning\venv\Lib\site-packages\langchain_core\runnables\utils.py:758: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Python 3.16; use inspect.iscoroutinefunction() instead
  and asyncio.iscoroutinefunction(func.__call__)




> Entering new AgentExecutor chain...

Invoking: `duckduckgo_search` with `{'query': 'capital of Madhya Pradesh'}`


India is a federal union comprising 28 states and 8 union territories, for a total of 36 subnational entities. Itscapitalis Bhopal, andMadhyaPradeshis the second-largest Indian state by area and the fifth-largest state by population with over 72 million residents. It borders the states of UttarPradesh, Chhattisgarh, Maharashtra, Gujarat, and Rajasthan. While most know it as thecapitalcityofMadhyaPradesh, Bhopal district is much more — a complete destination that invites tourists of every kind. Whether you're a history enthusiast, a nature lover, a foodie, or a culture buff, Bhopal has a story for you. ThecapitalofMadhyaPradesh, Bhopal situated in the heart of India, boasts a fascinating confluence of historicity and modernity. It is situated on the site of an 11th century city, Bhojapal , founded by Raja Bhoj. Bhopal today presents a multi-faceted profile... Nagaland b